# DocTamper 蒸馏训练（本机 / WSL / Linux）

使用变量 **`PROJECT_ROOT`** 表示 DocTamper 仓库根目录；数据与权重的落盘位置与 Colab 版 `cp` 目标一致（`{PROJECT_ROOT}/DocTamperV1-TrainingSet`、`pks/`、`pths/` 等），其余路径均相对 `PROJECT_ROOT` 书写。

## 1. 设置工程根目录并进入 `models`

将下面代码中的 **`PROJECT_ROOT`** 改为你本机 DocTamper 路径（对应 Colab 的 `/content/DocTamper`）。若启动 Jupyter 前已在仓库根目录 `cd`，可改用注释里的自动检测。

In [ ]:
from pathlib import Path
import os

# 仓库根目录 —— 请改为你的本机路径（与 Colab 中 /content/DocTamper 对应）
PROJECT_ROOT = Path("/home/nanxin/workspace/doctamper/DocTamper").resolve()

# 若始终在仓库根目录启动 Jupyter，可改为：
# PROJECT_ROOT = Path.cwd().resolve()
# assert (PROJECT_ROOT / "train_distill.py").exists(), "请修正 PROJECT_ROOT 或先 cd 到仓库根目录"

os.chdir(PROJECT_ROOT / "models")
print("PROJECT_ROOT =", PROJECT_ROOT)
print("cwd       =", Path.cwd())

## 2. 安装依赖

在 **`PROJECT_ROOT/models`** 下安装（与 DTD 笔记本一致）。需已安装 PyTorch；此处安装其余 Python 包并从源码编译 **jpegio**（在 `models` 目录内克隆构建）。

In [ ]:
!pip install -q lmdb albumentations segmentation_models_pytorch timm efficientnet_pytorch tqdm
!pip install -q opencv-python-headless Pillow

# jpegio 从源码安装（与根目录 ipynb 一致）；已存在目录则跳过 clone
from pathlib import Path
import subprocess

if not Path("jpegio").is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/dwgoon/jpegio.git"],
        check=True,
    )
%cd jpegio
!python setup.py install -q
%cd ..

## 3. 从外部目录拷贝数据与权重

对应 Colab：`TargetFolder` → 本机 **`TARGET_FOLDER`**（按需修改路径）。其中应包含：

- `DocTamperV1-TrainingSet`：LMDB 训练集（与 Colab 一致，将拷到 `{PROJECT_ROOT}/DocTamperV1-TrainingSet`）
- `checkpoints`：教师权重等（拷到 `{PROJECT_ROOT}/pths/checkpoints`，再复制 `dtd_doctamper.pth` 到 `{PROJECT_ROOT}/pths/`）

**`qt_table.pk`** 需已在仓库根目录 `{PROJECT_ROOT}/qt_table.pk`（Colab 版同样要求在工程根下）。首次使用该 TrainingSet 时会运行 `generate_pks.py`，产物为 `{PROJECT_ROOT}/pks/...`。

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 外部数据目录（对应 Colab：/content/drive/MyDrive/TargetFolder）
TARGET_FOLDER = Path.home() / "TargetFolder"

LMDB_NAME = "DocTamperV1-TrainingSet"
lmdb_dst = PROJECT_ROOT / LMDB_NAME
src_lmdb = TARGET_FOLDER / LMDB_NAME

assert src_lmdb.is_dir(), f"未找到 LMDB 目录: {src_lmdb}（请设置 TARGET_FOLDER）"
if lmdb_dst.exists():
    print(f"已存在，跳过拷贝: {lmdb_dst}")
else:
    shutil.copytree(src_lmdb, lmdb_dst)

qt_pk = PROJECT_ROOT / "qt_table.pk"
assert qt_pk.is_file(), f"缺少 qt_table.pk，请放到: {qt_pk}"

pks_path = PROJECT_ROOT / "pks" / f"{LMDB_NAME}_75.pk"
if not pks_path.is_file():
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "generate_pks.py"),
            "--lmdb_path",
            LMDB_NAME,
            "--minq",
            "75",
        ],
        cwd=str(PROJECT_ROOT),
        check=True,
    )
assert pks_path.is_file(), f"pks 生成失败: {pks_path}"

pths_dir = PROJECT_ROOT / "pths"
pths_dir.mkdir(parents=True, exist_ok=True)
src_ckpt = TARGET_FOLDER / "checkpoints"
dst_ckpt = pths_dir / "checkpoints"
if src_ckpt.is_dir():
    if dst_ckpt.exists():
        shutil.rmtree(dst_ckpt)
    shutil.copytree(src_ckpt, dst_ckpt)
    teacher_inner = dst_ckpt / "dtd_doctamper.pth"
    teacher_top = pths_dir / "dtd_doctamper.pth"
    if teacher_inner.is_file():
        shutil.copy2(teacher_inner, teacher_top)
else:
    assert (pths_dir / "dtd_doctamper.pth").is_file(), (
        f"未找到 {src_ckpt}，且 {pths_dir / 'dtd_doctamper.pth'} 也不存在"
    )

print("Data and checkpoints ready.")

## 4. 启动蒸馏训练

使用 `DocTamperV1-TrainingSet`（含 CM+SP+GE 全部篡改类型）。`--data_root`、`--teacher_pth`、`--save_dir` 均基于 **`PROJECT_ROOT`** 的相对布局，与 Colab 版一致。

In [ ]:
root = str(PROJECT_ROOT)
!CUDA_VISIBLE_DEVICES=0 python {root}/train_distill.py \
  --data_root {root} \
  --lmdb_name DocTamperV1-TrainingSet \
  --teacher_pth {root}/pths/dtd_doctamper.pth \
  --save_dir {root}/pths \
  --batch_size 96 \
  --num_workers 8 \
  --epochs 50
  # 可选：同步备份目录（对应 Colab 的 --save_dir_drive）
  # --save_dir_drive {root}/pths/backup_ckpts
  #--resume {root}/pths/light_dtd_distill_epoch20.pth

## 5. （可选）备份 checkpoint 到其他目录

对应 Colab：将 `light_dtd_distill_*.pth` 拷到网盘；本机可改为任意 **`BACKUP_DIR`**（与 `PROJECT_ROOT` 无关时需写绝对路径）。

In [ ]:
from pathlib import Path
import shutil

BACKUP_DIR = Path.home() / "DocTamper_ckpts"  # 按需修改
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

src_dir = PROJECT_ROOT / "pths"
n = 0
for p in sorted(src_dir.glob("light_dtd_distill_*.pth")):
    shutil.copy2(p, BACKUP_DIR / p.name)
    print("copied ->", BACKUP_DIR / p.name)
    n += 1
if n == 0:
    print("未找到 light_dtd_distill_*.pth")